# Train &amp; Evaluate the Play Type Classifier

LightGBM multiclass classifier predicting `P(run), P(pass), P(punt), P(field_goal)` from pre-snap game state, previous-play, and expanding coach-tendency features. See `src/fb_models/modeling/plays.py` (shared feature engineering) and `src/fb_models/modeling/play_type/` (this model's feature selection + train/predict).

In [18]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, log_loss

from fb_models.dataset import load_pbp_dataset, load_participation_dataset, load_games_dataset
from fb_models.modeling.plays import build_plays
from fb_models.modeling.play_type.features import PLAY_TYPE_FEATURE_COLS, select_play_type_features
from fb_models.modeling.play_type.train import train_play_type_classifier
from fb_models.modeling.play_type.predict import predict_play_type_probs

In [19]:
# Season-based split, matching the old evaluate.py convention: train on everything
# before TEST_SEASON, hold out TEST_SEASON entirely. Loading the full range up front
# (rather than loading train/test separately) matters here — the expanding
# coach-tendency and previous-play features need continuous history leading into
# the test season to be computed correctly.
SEASONS = list(range(2019, 2025))
TEST_SEASON = 2024

In [20]:
pbp = load_pbp_dataset(seasons=SEASONS)
participation = load_participation_dataset(seasons=SEASONS)
games = load_games_dataset()

plays = build_plays(pbp, participation, games)
plays.shape

(221042, 86)

In [21]:
train_plays = plays[plays["season"] < TEST_SEASON]
test_plays = plays[plays["season"] == TEST_SEASON]

train_df = select_play_type_features(train_plays)
test_df = select_play_type_features(test_plays)

print(f"train: {len(train_df):,} plays (seasons {SEASONS[0]}-{TEST_SEASON - 1})")
print(f"test:  {len(test_df):,} plays (season {TEST_SEASON})")

print(train_df.columns.tolist)

train: 184,069 plays (seasons 2019-2023)
test:  36,973 plays (season 2024)
<bound method IndexOpsMixin.tolist of Index(['down', 'ydstogo', 'yardline_100', 'goal_to_go', 'qtr',
       'game_seconds_remaining', 'half_seconds_remaining',
       'score_differential', 'posteam_timeouts_remaining',
       'defteam_timeouts_remaining', 'red_zone', 'goal_line', 'two_minute',
       'four_minute', 'passing_down', 'short_yardage', 'neutral_script',
       'previous_yards_gained', 'previous_first_down', 'previous_turnover',
       'off_pass_rate_hist', 'off_early_down_pass_rate_hist',
       'off_red_zone_pass_rate_hist', 'off_goal_line_pass_rate_hist',
       'off_neutral_pass_rate_hist', 'off_fourth_down_go_for_it_rate_hist',
       'off_shotgun_rate_hist', 'off_no_huddle_rate_hist',
       'def_pass_rate_allowed_hist', 'def_blitz_rate_hist',
       'def_pressure_rate_hist', 'def_man_rate_hist', 'posteam_spread_line',
       'total_line', 'div_game', 'posteam_rest', 'defteam_rest', 'temp',
    

In [22]:
clf = train_play_type_classifier(train_df)
clf.classes_

array(['field_goal', 'pass', 'punt', 'run'], dtype=object)

## Evaluation on the held-out season

In [23]:
X_test = test_df[PLAY_TYPE_FEATURE_COLS]
y_test = test_df["play_type"]

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

accuracy = (y_pred == y_test.to_numpy()).mean()
loss = log_loss(y_test, y_proba, labels=clf.classes_)

print(f"accuracy: {accuracy:.3f}")
print(f"log_loss: {loss:.3f}")
print()
print(classification_report(y_test, y_pred, labels=clf.classes_, zero_division=0))

accuracy: 0.719
log_loss: 0.551

              precision    recall  f1-score   support

  field_goal       0.79      0.96      0.87      1115
        pass       0.79      0.64      0.71     19125
        punt       0.91      0.99      0.95      2046
         run       0.63      0.77      0.69     14687

    accuracy                           0.72     36973
   macro avg       0.78      0.84      0.80     36973
weighted avg       0.73      0.72      0.72     36973



In [24]:
labels = list(clf.classes_)
cm = confusion_matrix(y_test, y_pred, labels=labels)
pd.DataFrame(cm, index=labels, columns=labels)

,field_goal,pass,punt,run
field_goal,1071,12,21,11
pass,189,12200,109,6627
punt,10,8,2025,3
run,80,3244,74,11289


### Calibration sanity check

Per the design spec, calibration matters more than raw accuracy here — the simulation samples from the predicted distribution rather than taking the argmax, so mean predicted probabilities should track real-world marginal rates (e.g. overall pass rate ~55-58% in the modern NFL).

In [25]:
mean_predicted_probs = pd.Series(y_proba.mean(axis=0), index=labels, name="mean_predicted_prob")
actual_rates = y_test.value_counts(normalize=True).reindex(labels).rename("actual_rate")

pd.concat([mean_predicted_probs, actual_rates], axis=1)

,mean_predicted_prob,actual_rate
field_goal,0.035911,0.030157
pass,0.463366,0.517269
punt,0.059428,0.055338
run,0.441296,0.397236


### Game-context features (weather, betting lines, rest)

Added `posteam_spread_line`/`total_line` (pregame game-script expectation), `div_game`, `posteam_rest`/`defteam_rest`, and `roof`/`temp`/`wind` from the new `fs.games` data. `away_score`/`home_score` are deliberately excluded — those are the game's final outcome and would leak straight into a pre-snap feature. Checking where these land in feature importance below.

In [26]:
game_context_cols = [
    "posteam_spread_line", "total_line", "div_game",
    "posteam_rest", "defteam_rest", "roof", "temp", "wind",
]

importances = pd.Series(clf.feature_importances_, index=clf.feature_name_)
importances_pct = (importances / importances.sum() * 100).sort_values(ascending=False).round(2)

ranks = pd.Series(
    {col: importances_pct.index.get_loc(col) + 1 for col in game_context_cols},
    name="rank",
).sort_values()
pd.concat([ranks, importances_pct.rename("pct_of_total_splits")], axis=1, join="inner").loc[ranks.index]

,rank,pct_of_total_splits
temp,17,2.77
total_line,18,2.57
posteam_spread_line,19,2.48
wind,23,1.89
posteam_rest,28,0.69
defteam_rest,29,0.67
div_game,33,0.17
roof,37,0.10


### `previous_epa` / `previous_success`: tested and dropped

Both would have needed a value the simulation can't produce on its own — EPA is the output of a separate expected-points model we haven't built, and "success" is conventionally EPA-derived too. Before committing to that scope, we measured whether they were actually earning their keep: `previous_epa` ranked 5th of 37 features by LightGBM split count, but `previous_success` ranked dead last (~0% of splits, redundant with `previous_epa`/`previous_yards_gained`/`previous_first_down`) — and a direct ablation (retrain without both, compare held-out accuracy/log_loss) showed dropping both cost only 0.00065 accuracy and 0.0027 log loss. High importance under the full feature set didn't imply high marginal value: LightGBM was substituting `previous_yards_gained`, `previous_first_down`, and the coach-tendency features when EPA wasn't available. Not a good trade for building a whole EPA model, so both features were removed from `PLAY_TYPE_NUMERIC_COLS` (and their source columns dropped from `PBP_RETAIN_COLS`/`plays.py`'s lag computation) rather than kept.

## Example predictions

`predict_play_type_probs` takes a single-row DataFrame (not a raw numpy array — `previous_play_type`/`score_margin_bucket`/`yardage_bucket` are pandas categoricals that LightGBM consumes natively) and returns the probability dict the simulation samples from. A few illustrative game-state scenarios below, holding everything except down/distance/field position/score constant.

In [27]:
situational_cols = ["down", "ydstogo", "yardline_100", "goal_to_go", "score_differential"]

scenarios = {
    "1st & 10, midfield, tied": [1, 10, 50, 0, 0],
    "3rd & long, down 3": [3, 12, 60, 0, 0],
    "4th & short, midfield, down 3": [4, 1, 50, 0, -3],
    "4th & long, own territory": [4, 8, 75, 0, 0],
    "4th & short, field goal range": [4, 2, 25, 0, -2],
    "Goal to go, 2nd down": [2, 3, 3, 1, 3],
}

rows = []
for name, values in scenarios.items():
    sample_play = test_df.iloc[[0]].copy()
    sample_play.loc[:, situational_cols] = values

    probs = predict_play_type_probs(clf, sample_play)
    rows.append({
        "scenario": name,
        "prob_run": probs["run"],
        "prob_pass": probs["pass"],
        "prob_punt": probs["punt"],
        "prob_fieldgoal": probs["field_goal"],
    })

sample_plays = pd.DataFrame(rows).set_index("scenario")
sample_plays[["prob_run", "prob_pass", "prob_punt", "prob_fieldgoal"]]

,prob_run,prob_pass,prob_punt,prob_fieldgoal
scenario,,,,
"1st & 10, midfield, tied",0.605147,0.394806,0.000013,3.379399e-05
"3rd & long, down 3",0.150655,0.849329,0.000010,6.432216e-06
"4th & short, midfield, down 3",0.577820,0.232114,0.189696,3.699774e-04
"4th & long, own territory",0.000179,0.000431,0.999389,5.815171e-07
"4th & short, field goal range",0.099276,0.157382,0.000164,7.431773e-01
"Goal to go, 2nd down",0.654131,0.345821,0.000007,4.120854e-05


## Save artifact

Retrains on the full dataset (no held-out season) before saving, so the shipped model benefits from all available history.

In [28]:
import joblib

from fb_models.config import MODELS_DIR

full_df = select_play_type_features(plays)
final_clf = train_play_type_classifier(full_df)

MODELS_DIR.mkdir(exist_ok=True)

artifact_path = MODELS_DIR / "play_type_classifier.joblib"
joblib.dump(final_clf, artifact_path)
print(f"saved {artifact_path} ({len(full_df):,} plays, seasons {SEASONS[0]}-{SEASONS[-1]})")

saved /home/davis/projects/fb_models/models/play_type_classifier.joblib (221,042 plays, seasons 2019-2024)
